# Polars Expressions

This notebook demonstrates the `mintalib.expressions` module — polars expression factory functions for use with `select` and `with_columns`.

Expressions are named in **upper case** (e.g. `SMA`, `EMA`, `MACD`). An optional `src` keyword parameter sets the source expression; by default series-based indicators use `pl.col("close")`. Multi-column outputs like `MACD` return a tuple of expressions. Expressions can be chained using `.pipe()`.

In [22]:
import polars as pl
from mintalib.expressions import SMA, EMA, ATR, MACD, ROC

In [23]:
from mintalib.samples import sample_prices

prices = sample_prices(backend="polars")
prices


date,open,high,low,close,volume
datetime[ns],f64,f64,f64,f64,i64
1980-12-12 00:00:00,0.098943,0.099373,0.098943,0.098943,469033600
1980-12-15 00:00:00,0.094211,0.094211,0.093781,0.093781,175884800
1980-12-16 00:00:00,0.087328,0.087328,0.086898,0.086898,105728000
1980-12-17 00:00:00,0.089049,0.089479,0.089049,0.089049,86441600
1980-12-18 00:00:00,0.09163,0.092061,0.09163,0.09163,73449600
…,…,…,…,…,…
2024-10-15 00:00:00,233.610001,237.490005,232.369995,233.850006,64751400
2024-10-16 00:00:00,231.600006,232.119995,229.839996,231.779999,34082200
2024-10-17 00:00:00,233.429993,233.850006,230.520004,232.149994,32993800


In [24]:
prices.select(
    SMA(20).alias("ema20")
)


ema20
f64
null
null
null
null
null
…
227.524
228.0785
228.2425


In [25]:
prices.select(
    ATR(14).alias("atr")
)



atr
f64
null
null
null
null
null
…
4.516121
4.479971
4.39783


In [26]:
prices.select(
    MACD()
)

macd,macdsignal,macdhist
f64,f64,f64
null,null,null
null,null,null
null,null,null
null,null,null
null,null,null
…,…,…
1.815958,1.313965,0.501993
1.941114,1.439395,0.501719
2.046565,1.560829,0.485736


In [27]:
# Expression can be piped into other expression factory functions.
# The first expression argument is passed as `src` to the next function.

prices.select(
    EMA(20).pipe(ROC, 1)
)

roc
f64
null
null
null
null
null
…
0.003006
0.001845
0.001821


In [28]:
prices.select(
    MACD(),
    sma=SMA(50),
    atr=ATR(14),
    trend=EMA(50).pipe(ROC, 1)
)

macd,macdsignal,macdhist,sma,atr,trend
f64,f64,f64,f64,f64,f64
null,null,null,null,null,null
null,null,null,null,null,null
null,null,null,null,null,null
null,null,null,null,null,null
null,null,null,null,null,null
…,…,…,…,…,…
1.815958,1.313965,0.501993,224.138625,4.516121,0.001811
1.941114,1.439395,0.501719,224.634417,4.479971,0.001374
2.046565,1.560829,0.485736,225.085868,4.39783,0.001383
